In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Bus RavKav - PM-peak and midday journeys by TAZ (step 8 for the other periods)

The same processing as `BusRavKav_matrix.ipynb` (step 8: stop-to-TAZ tagging, boarding events deduplicated, journeys linked by `bus_trip_id`, weights `total_boardings`,
average of the four Tuesdays 2022-05-03 / 17 / 24 / 31), run for three windows of the journey's start hour (`bus_trip_hour`): **AM 06:00-09:00** (hours 6, 7, 8; must reproduce
the committed step-8 files exactly), **PM peak 16:00-19:00** (16, 17, 18) and **midday 10:00-14:00** (10, 11, 12, 13), plus the hourly profile of journeys over the whole day
inside the study area. The stop-to-TAZ tags are those of step 8 (`Output/bus/bus_stops_taz.csv`). The outputs feed the survey-based PM and midday matrices (steps 15 and 16 with `NOFIT_PERIOD`) and the
time-of-day test of the MoT validation (T8) for the two periods.

In [2]:
import os
import numpy as np, pandas as pd
FILES = [f'Input/BusRavKav/May_2022/trips_table_2022-05-{d}.csv' for d in ['03', '17', '24', '31']]
for f in FILES: assert not open(f, 'rb').read(40).startswith(b'version https://git-lfs'), f'{f} is an LFS pointer: python3 tools/lfs_pull.py {f}'
USECOLS = ['passanger_trip_id', 'bus_trip_id', 'weekday', 'bus_trip_hour', 'total_boardings', 'orig_stop_code', 'dest_stop_code', 'board_stop_code', 'alight_stop_code']
PERIODS = {'AM': [6, 7, 8], 'PM': [16, 17, 18], 'MD': [10, 11, 12, 13]}
st = pd.read_csv('Output/bus/bus_stops_taz.csv'); stop_to_taz = st.drop_duplicates('stop_code').set_index('stop_code')['TAZ_NUMBER']
OUTP = 'Output/bus/periods'; os.makedirs(OUTP, exist_ok=True)

In [3]:
od_days = {p: [] for p in PERIODS}; board_days = {p: [] for p in PERIODS}; alight_days = {p: [] for p in PERIODS}; hourly = []
for f in FILES:
    df = pd.read_csv(f, usecols=USECOLS); df = df[df['weekday'] == 3]
    legs = df.drop_duplicates(['passanger_trip_id', 'bus_trip_id']); journeys = legs.drop_duplicates('bus_trip_id').copy()
    legs = legs.assign(b_taz=legs['board_stop_code'].map(stop_to_taz), a_taz=legs['alight_stop_code'].map(stop_to_taz))
    journeys['o_taz'] = journeys['orig_stop_code'].map(stop_to_taz); journeys['d_taz'] = journeys['dest_stop_code'].map(stop_to_taz)
    inside = journeys.dropna(subset=['o_taz', 'd_taz'])
    hourly.append(pd.DataFrame({'journeys, both ends in TAZ_North': inside.groupby('bus_trip_hour')['total_boardings'].sum(),
                                'journeys, all': journeys.groupby('bus_trip_hour')['total_boardings'].sum(),
                                'boarding legs in TAZ_North': legs.dropna(subset=['b_taz']).groupby('bus_trip_hour')['total_boardings'].sum()}))
    for p, hrs in PERIODS.items():
        lp = legs[legs['bus_trip_hour'].isin(hrs)]; jp = inside[inside['bus_trip_hour'].isin(hrs)]
        board_days[p].append(lp.dropna(subset=['b_taz']).groupby('b_taz')['total_boardings'].sum())
        alight_days[p].append(lp.dropna(subset=['a_taz']).groupby('a_taz')['total_boardings'].sum())
        od_days[p].append(jp.groupby(['o_taz', 'd_taz'])['total_boardings'].sum())
    print(f"{f.split('/')[-1]}: journeys inside TAZ_North " + ', '.join(f"{p} {od_days[p][-1].sum():,.0f}" for p in PERIODS))
    del df, legs, journeys, inside

trips_table_2022-05-03.csv: journeys inside TAZ_North AM 86,486, PM 73,650, MD 112,171


trips_table_2022-05-17.csv: journeys inside TAZ_North AM 95,896, PM 83,014, MD 124,032


trips_table_2022-05-24.csv: journeys inside TAZ_North AM 97,097, PM 80,707, MD 123,373


trips_table_2022-05-31.csv: journeys inside TAZ_North AM 97,334, PM 82,686, MD 127,765


In [4]:
for p in PERIODS:
    od = pd.concat(od_days[p], axis=1).fillna(0).mean(axis=1).unstack().fillna(0); od.index = od.index.astype(int); od.columns = od.columns.astype(int); od.index.name = 'orig_taz'
    od.to_csv(f'{OUTP}/bus_od_taz_avg_{p}.csv', float_format='%.6g')
    ba = pd.concat([pd.concat(board_days[p], axis=1).fillna(0).mean(axis=1).rename('avg_boardings'), pd.concat(alight_days[p], axis=1).fillna(0).mean(axis=1).rename('avg_alightings')], axis=1).fillna(0)
    ba.index = ba.index.astype(int); ba.index.name = 'TAZ_NUMBER'; ba.to_csv(f'{OUTP}/bus_boardings_alightings_taz_{p}.csv', float_format='%.6g')
    print(f'{p}: OD {od.shape}, {od.values.sum():,.0f} journeys per average Tuesday; boardings {ba.avg_boardings.sum():,.0f}')
hp = pd.concat(hourly).groupby(level=0).sum() / len(FILES); hp.index.name = 'bus_trip_hour'; hp.to_csv(f'{OUTP}/bus_hourly_journeys_2022.csv', float_format='%.1f')
print(hp.round(0).to_string())

AM: OD (722, 711), 94,203 journeys per average Tuesday; boardings 157,264


PM: OD (717, 709), 80,014 journeys per average Tuesday; boardings 122,174


MD: OD (722, 720), 121,835 journeys per average Tuesday; boardings 210,809
               journeys, both ends in TAZ_North  journeys, all  boarding legs in TAZ_North
bus_trip_hour                                                                             
0                                         366.0         7825.0                       447.0
1                                         109.0          668.0                       138.0
2                                          57.0          137.0                        72.0
3                                          66.0          148.0                       108.0
4                                         484.0         1906.0                       980.0
5                                        6207.0        32309.0                     24168.0
6                                       18168.0       106172.0                     36637.0
7                                       40924.0       225995.0                     63146.0
8              

## Regression: the AM files must equal the committed step-8 files

In [5]:
for new, old in [(f'{OUTP}/bus_od_taz_avg_AM.csv', 'Output/bus/bus_od_taz_avg.csv'), (f'{OUTP}/bus_boardings_alightings_taz_AM.csv', 'Output/bus/bus_boardings_alightings_taz.csv')]:
    a = pd.read_csv(new, index_col=0); b = pd.read_csv(old, index_col=0)
    a.index = a.index.astype(int); b.index = b.index.astype(int); a.columns = a.columns.astype(str); b.columns = b.columns.astype(str)
    idx = a.index.union(b.index); cols = a.columns.union(b.columns)
    diff = (a.reindex(index=idx, columns=cols).fillna(0) - b.reindex(index=idx, columns=cols).fillna(0)).abs().values.max()
    print(f'{os.path.basename(new)} vs {old}: max absolute difference {diff:.2e}'); assert diff < 1e-3

bus_od_taz_avg_AM.csv vs Output/bus/bus_od_taz_avg.csv: max absolute difference 0.00e+00
bus_boardings_alightings_taz_AM.csv vs Output/bus/bus_boardings_alightings_taz.csv: max absolute difference 0.00e+00
